# 08 — Transformation-Based Synthetic-Augmentation Classification

## Objective

This notebook evaluates whether the validated transformation-based synthetic
dataset improves drone–bird micro-Doppler classification under limited real-data
conditions.

The experiment combines:

- 1,150 samples from the balanced 10% real training subset;
- 1,150 validated synthetic samples derived from that subset;
- 2,300 balanced training observations in total, derived from 1,150 independent real parent samples.

The official real validation and test partitions remain unchanged. Synthetic
samples are used exclusively during model training.

The augmented model uses the same CNN architecture, optimizer, loss function,
random seed, callbacks, threshold-selection rule, and evaluation metrics as the
real-only baselines.

The augmented result will be compared with:

1. The 10% real-only baseline.
2. The 25% real-only baseline.
The primary question is whether 10% real data with synthetic augmentation can
approach the 25% real-only baseline.

## 1. Safe Execution and Reproducibility

This notebook defaults to `RUN_TRAINING = False`. After a kernel restart, running all cells loads the saved best checkpoint and training history, then reproduces the validation and test analyses without fitting the model again. Training is authorized only when the flag is deliberately changed and the protected output paths are empty. Fixed random seeds and deterministic TensorFlow operations are used where available.


In [ ]:
import hashlib
import json
import random

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import tensorflow as tf

from pathlib import Path

from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    roc_auc_score,
    recall_score
)

from tensorflow.keras import (
    layers,
    models,
    regularizers
)

In [ ]:
RANDOM_SEED = 42
BATCH_SIZE = 64
MAX_EPOCHS = 50

REAL_SUBSET = "10_percent"
SYNTHETIC_RATIO = 1.0

EXPERIMENT_NAME = (
    "10_percent_real_plus_"
    "synthetic_1_to_1_seed_42"
)

# Safe resume default: load the completed checkpoint without retraining.
# Change to True only for a new, intentional experiment with new paths.
RUN_TRAINING = False

random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)
tf.random.set_seed(RANDOM_SEED)

try:
    tf.config.experimental.enable_op_determinism()
    print(
        "TensorFlow deterministic operations enabled."
    )
except Exception as error:
    print(
        "Deterministic operations unavailable:",
        error
    )

print("TensorFlow version:", tf.__version__)
print(
    "Available GPUs:",
    tf.config.list_physical_devices("GPU")
)
print("Experiment:", EXPERIMENT_NAME)
print("Run training:", RUN_TRAINING)

In [ ]:
OFFICIAL_DATA_DIR = Path(
    "../data/processed/official_split"
)

LIMITED_DATA_DIR = Path(
    "../data/processed/limited_subsets"
)

SYNTHETIC_DATASET_DIR = (
    Path("../data/processed/synthetic_subsets")
    / (
        "10_percent_signal_augmentation_"
        "ratio_1_seed_42"
    )
)

BASELINE_OUTPUT_DIR = Path(
    "../outputs/baseline_classification"
)

OUTPUT_DIR = (
    Path(
        "../outputs/"
        "synthetic_augmentation_classification"
    )
    / EXPERIMENT_NAME
)

CHECKPOINT_PATH = (
    Path(
        "../checkpoints/"
        "synthetic_augmentation"
    )
    / f"{EXPERIMENT_NAME}.keras"
)

required_files = [
    OFFICIAL_DATA_DIR / "X_train.npy",
    OFFICIAL_DATA_DIR / "y_train.npy",
    OFFICIAL_DATA_DIR / "X_validation.npy",
    OFFICIAL_DATA_DIR / "y_validation.npy",
    OFFICIAL_DATA_DIR / "X_test.npy",
    OFFICIAL_DATA_DIR / "y_test.npy",
    OFFICIAL_DATA_DIR / "metadata_test.csv",
    LIMITED_DATA_DIR / "indices_10_percent.npy",
    SYNTHETIC_DATASET_DIR / "X_synthetic.npy",
    SYNTHETIC_DATASET_DIR / "y_synthetic.npy",
    SYNTHETIC_DATASET_DIR
    / "metadata_synthetic.csv",
    SYNTHETIC_DATASET_DIR
    / "generation_manifest.json",
    BASELINE_OUTPUT_DIR
    / "10_percent_seed_42"
    / "test_metrics.csv",
    BASELINE_OUTPUT_DIR
    / "25_percent_seed_42"
    / "test_metrics.csv"
]

missing_files = [
    path
    for path in required_files
    if not path.exists()
]

if missing_files:
    raise FileNotFoundError(
        "Missing required files:\n"
        + "\n".join(
            str(path.resolve())
            for path in missing_files
        )
    )

CHECKPOINT_PATH.parent.mkdir(
    parents=True,
    exist_ok=True
)

print("All required files were found.")
print(
    "Synthetic dataset:",
    SYNTHETIC_DATASET_DIR.resolve()
)
print(
    "Result directory:",
    OUTPUT_DIR.resolve()
)
print(
    "Checkpoint:",
    CHECKPOINT_PATH.resolve()
)
print(
    "Existing checkpoint:",
    CHECKPOINT_PATH.exists()
)

In [ ]:
if RUN_TRAINING:
    protected_paths = [
        CHECKPOINT_PATH,
        OUTPUT_DIR / "test_metrics.csv",
        OUTPUT_DIR / "training_history.csv"
    ]

    existing_protected_paths = [
        path
        for path in protected_paths
        if path.exists()
    ]

    if existing_protected_paths:
        raise FileExistsError(
            "Existing augmented experiment "
            "artifacts were found:\n"
            + "\n".join(
                str(path.resolve())
                for path
                in existing_protected_paths
            )
            + "\nSet RUN_TRAINING = False "
              "to load the completed experiment."
        )

    # The directory itself may already exist.
    # This does not overwrite any files.
    OUTPUT_DIR.mkdir(
        parents=True,
        exist_ok=True
    )

    print(
        "No completed training artifacts "
        "were found. Training is authorized."
    )

else:
    required_saved_artifacts = [
        CHECKPOINT_PATH,
        OUTPUT_DIR / "training_history.csv"
    ]

    missing_saved_artifacts = [
        path
        for path in required_saved_artifacts
        if not path.exists()
    ]

    if missing_saved_artifacts:
        raise FileNotFoundError(
            "RUN_TRAINING is False, but "
            "required saved artifacts are missing:\n"
            + "\n".join(
                str(path.resolve())
                for path
                in missing_saved_artifacts
            )
        )

    print(
        "SAFE RESUME MODE: the saved checkpoint and "
        "training history will be loaded. Training is disabled."
    )

print(
    "Experiment protection checks passed."
)

## 2. Data Loading and Source Verification

The balanced 10% real subset, validated synthetic dataset, and untouched official validation and test partitions are loaded and checked before model preparation.


In [ ]:
X_train_complete = np.load(
    OFFICIAL_DATA_DIR / "X_train.npy",
    mmap_mode="r"
)

y_train_complete = np.load(
    OFFICIAL_DATA_DIR / "y_train.npy",
    mmap_mode="r"
)

source_indices = np.load(
    LIMITED_DATA_DIR
    / "indices_10_percent.npy"
)

X_real = np.asarray(
    X_train_complete[source_indices],
    dtype=np.float32
)

y_real = np.asarray(
    y_train_complete[source_indices],
    dtype=np.uint8
)

X_synthetic = np.asarray(
    np.load(
        SYNTHETIC_DATASET_DIR
        / "X_synthetic.npy",
        mmap_mode="r"
    ),
    dtype=np.float32
)

y_synthetic = np.asarray(
    np.load(
        SYNTHETIC_DATASET_DIR
        / "y_synthetic.npy",
        mmap_mode="r"
    ),
    dtype=np.uint8
)

metadata_synthetic = pd.read_csv(
    SYNTHETIC_DATASET_DIR
    / "metadata_synthetic.csv"
)

with open(
    SYNTHETIC_DATASET_DIR
    / "generation_manifest.json",
    "r",
    encoding="utf-8"
) as file:
    synthetic_manifest = json.load(file)

X_validation = np.asarray(
    np.load(
        OFFICIAL_DATA_DIR
        / "X_validation.npy",
        mmap_mode="r"
    ),
    dtype=np.float32
)

y_validation = np.asarray(
    np.load(
        OFFICIAL_DATA_DIR
        / "y_validation.npy",
        mmap_mode="r"
    ),
    dtype=np.uint8
)

X_test = np.asarray(
    np.load(
        OFFICIAL_DATA_DIR
        / "X_test.npy",
        mmap_mode="r"
    ),
    dtype=np.float32
)

y_test = np.asarray(
    np.load(
        OFFICIAL_DATA_DIR
        / "y_test.npy",
        mmap_mode="r"
    ),
    dtype=np.uint8
)

metadata_test = pd.read_csv(
    OFFICIAL_DATA_DIR
    / "metadata_test.csv"
)

print(
    "Real training tensor:",
    X_real.shape
)

print(
    "Synthetic training tensor:",
    X_synthetic.shape
)

print(
    "Validation tensor:",
    X_validation.shape
)

print(
    "Test tensor:",
    X_test.shape
)

## 3. Augmented Training-Set Construction

The real and synthetic samples are combined at a 1:1 ratio. The resulting training set contains 2,300 observations, balanced equally between birds and drones. Synthetic children remain linked to 1,150 real parents and therefore are not treated as independent real observations.


In [ ]:
X_train_augmented = np.concatenate(
    [
        X_real,
        X_synthetic
    ],
    axis=0
)

y_train_augmented = np.concatenate(
    [
        y_real,
        y_synthetic
    ],
    axis=0
)

training_source = np.concatenate([
    np.repeat("real", len(X_real)),
    np.repeat(
        "synthetic",
        len(X_synthetic)
    )
])

assert X_real.shape == (
    1150,
    5,
    150
)

assert X_synthetic.shape == (
    1150,
    5,
    150
)

assert X_train_augmented.shape == (
    2300,
    5,
    150
)

assert y_train_augmented.shape == (
    2300,
)

assert np.array_equal(
    np.bincount(y_train_augmented),
    [1150, 1150]
)

assert np.isfinite(
    X_train_augmented
).all()

assert X_train_augmented.min() >= 0.0
assert X_train_augmented.max() <= 1.0

assert len(metadata_synthetic) == 1150

assert np.array_equal(
    y_synthetic,
    metadata_synthetic[
        "parent_label"
    ].to_numpy(dtype=np.uint8)
)

assert (
    metadata_synthetic[
        "official_training_index"
    ]
    .isin(source_indices)
    .all()
)

print(
    "Augmented training tensor:",
    X_train_augmented.shape
)

print(
    "Augmented class counts [bird, drone]:",
    np.bincount(y_train_augmented)
)

print(
    "Training-source counts:",
    pd.Series(
        training_source
    ).value_counts().to_dict()
)

print(
    "The augmented training dataset "
    "passed all validation checks."
)

## 4. TensorFlow Input Pipelines

A singleton channel dimension is added to each `(5, 150)` input. Training data are shuffled inside the `tf.data` pipeline, while validation and test data retain their fixed order.


In [ ]:
X_train_augmented_model = (
    X_train_augmented[
        ...,
        np.newaxis
    ]
)

X_validation_model = (
    X_validation[
        ...,
        np.newaxis
    ]
)

X_test_model = (
    X_test[
        ...,
        np.newaxis
    ]
)

training_dataset = (
    tf.data.Dataset
    .from_tensor_slices((
        X_train_augmented_model,
        y_train_augmented
    ))
    .shuffle(
        buffer_size=len(
            y_train_augmented
        ),
        seed=RANDOM_SEED,
        reshuffle_each_iteration=True
    )
    .batch(BATCH_SIZE)
    .prefetch(tf.data.AUTOTUNE)
)

validation_dataset = (
    tf.data.Dataset
    .from_tensor_slices((
        X_validation_model,
        y_validation
    ))
    .batch(BATCH_SIZE)
    .prefetch(tf.data.AUTOTUNE)
)

test_dataset = (
    tf.data.Dataset
    .from_tensor_slices((
        X_test_model,
        y_test
    ))
    .batch(BATCH_SIZE)
    .prefetch(tf.data.AUTOTUNE)
)

assert X_train_augmented_model.shape == (
    2300,
    5,
    150,
    1
)

assert X_validation_model.shape == (
    6988,
    5,
    150,
    1
)

assert X_test_model.shape == (
    6996,
    5,
    150,
    1
)

print(
    "Model training tensor:",
    X_train_augmented_model.shape
)

print(
    "Training batches:",
    len(training_dataset)
)

print(
    "Validation batches:",
    len(validation_dataset)
)

print(
    "Test batches:",
    len(test_dataset)
)

## 5. Fixed CNN Architecture

The architecture is identical to every real-only baseline, ensuring that differences are attributable to the training data rather than model capacity.


In [ ]:
def build_baseline_model(
    input_shape=(5, 150, 1)
):
    model = models.Sequential([
        layers.Input(
            shape=input_shape
        ),

        layers.Conv2D(
            filters=16,
            kernel_size=(3, 7),
            padding="same"
        ),
        layers.BatchNormalization(),
        layers.Activation("relu"),
        layers.MaxPooling2D(
            pool_size=(1, 2)
        ),

        layers.Conv2D(
            filters=32,
            kernel_size=(3, 5),
            padding="same"
        ),
        layers.BatchNormalization(),
        layers.Activation("relu"),
        layers.MaxPooling2D(
            pool_size=(1, 2)
        ),

        layers.Conv2D(
            filters=64,
            kernel_size=(3, 3),
            padding="same"
        ),
        layers.BatchNormalization(),
        layers.Activation("relu"),
        layers.MaxPooling2D(
            pool_size=(1, 2)
        ),

        layers.GlobalAveragePooling2D(),

        layers.Dense(
            units=32,
            activation="relu",
            kernel_regularizer=(
                regularizers.l2(1e-4)
            )
        ),

        layers.Dropout(0.30),

        layers.Dense(
            units=1,
            activation="sigmoid"
        )
    ])

    model.compile(
        optimizer=tf.keras.optimizers.Adam(
            learning_rate=1e-3
        ),
        loss="binary_crossentropy",
        metrics=[
            "accuracy",

            tf.keras.metrics.AUC(
                name="roc_auc",
                curve="ROC"
            ),

            tf.keras.metrics.AUC(
                name="pr_auc",
                curve="PR"
            ),

            tf.keras.metrics.Precision(
                name="precision"
            ),

            tf.keras.metrics.Recall(
                name="recall"
            )
        ]
    )

    return model

In [ ]:
tf.keras.backend.clear_session()

random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)
tf.random.set_seed(RANDOM_SEED)

model_augmented = build_baseline_model()

total_parameters = (
    model_augmented.count_params()
)

trainable_parameters = int(
    np.sum([
        np.prod(variable.shape)
        for variable
        in model_augmented.trainable_weights
    ])
)

non_trainable_parameters = int(
    np.sum([
        np.prod(variable.shape)
        for variable
        in model_augmented.non_trainable_weights
    ])
)

assert total_parameters == 29121
assert trainable_parameters == 28897
assert non_trainable_parameters == 224

print(
    "Total parameters:",
    total_parameters
)

print(
    "Trainable parameters:",
    trainable_parameters
)

print(
    "Non-trainable parameters:",
    non_trainable_parameters
)

print(
    "Architecture verified: it matches "
    "the real-only baselines."
)

model_augmented.summary()

## 6. Protected Training or Checkpoint Restoration

Early stopping and learning-rate reduction are defined for an authorized training run. In the default safe-resume mode, the best saved checkpoint and its training history are loaded without retraining.


In [ ]:
callbacks_augmented = [
    tf.keras.callbacks.EarlyStopping(
        monitor="val_loss",
        patience=8,
        restore_best_weights=True,
        verbose=1
    ),

    tf.keras.callbacks.ReduceLROnPlateau(
        monitor="val_loss",
        factor=0.5,
        patience=4,
        min_lr=1e-6,
        verbose=1
    ),

    tf.keras.callbacks.ModelCheckpoint(
        filepath=CHECKPOINT_PATH,
        monitor="val_loss",
        save_best_only=True,
        verbose=1
    ),

    tf.keras.callbacks.CSVLogger(
        OUTPUT_DIR / "training_log.csv"
    )
]

In [ ]:
if RUN_TRAINING:
    history_augmented = (
        model_augmented.fit(
            training_dataset,
            validation_data=(
                validation_dataset
            ),
            epochs=MAX_EPOCHS,
            callbacks=callbacks_augmented,
            verbose=1
        )
    )

    history_augmented_df = pd.DataFrame(
        history_augmented.history
    )

    history_augmented_df.insert(
        0,
        "epoch",
        np.arange(
            1,
            len(history_augmented_df) + 1
        )
    )

    history_augmented_df.to_csv(
        OUTPUT_DIR
        / "training_history.csv",
        index=False
    )

    print(
        "Training history saved immediately."
    )

else:
    model_augmented = (
        tf.keras.models.load_model(
            CHECKPOINT_PATH
        )
    )

    history_augmented_df = pd.read_csv(
        OUTPUT_DIR
        / "training_history.csv"
    )

    if (
        "epoch"
        not in history_augmented_df.columns
    ):
        history_augmented_df.insert(
            0,
            "epoch",
            np.arange(
                1,
                len(history_augmented_df) + 1
            )
        )

    print(
        "Saved augmented checkpoint loaded; "
        "no training was performed."
    )

assert (
    model_augmented.count_params()
    == 29121
)

In [ ]:
best_epoch = int(
    history_augmented_df.loc[
        history_augmented_df[
            "val_loss"
        ].idxmin(),
        "epoch"
    ]
)

best_validation_loss = float(
    history_augmented_df[
        "val_loss"
    ].min()
)

best_epoch_row = (
    history_augmented_df.loc[
        history_augmented_df[
            "val_loss"
        ].idxmin()
    ]
)

print(
    "Completed epochs:",
    len(history_augmented_df)
)

print(
    "Best epoch:",
    best_epoch
)

print(
    "Best validation loss:",
    best_validation_loss
)

print(
    "Best-epoch validation accuracy:",
    float(
        best_epoch_row[
            "val_accuracy"
        ]
    )
)

print(
    "Best-epoch validation ROC-AUC:",
    float(
        best_epoch_row[
            "val_roc_auc"
        ]
    )
)

print(
    "Checkpoint saved:",
    CHECKPOINT_PATH.exists()
)

In [ ]:
fig, axes = plt.subplots(
    1,
    3,
    figsize=(16, 4),
    constrained_layout=True
)

plot_configuration = [
    (
        "loss",
        "val_loss",
        "Binary Cross-Entropy Loss",
        "Loss"
    ),
    (
        "accuracy",
        "val_accuracy",
        "Accuracy",
        "Accuracy"
    ),
    (
        "roc_auc",
        "val_roc_auc",
        "ROC-AUC",
        "ROC-AUC"
    )
]

for axis, (
    training_metric,
    validation_metric,
    title,
    y_label
) in zip(
    axes,
    plot_configuration
):
    axis.plot(
        history_augmented_df["epoch"],
        history_augmented_df[
            training_metric
        ],
        label="Training"
    )

    axis.plot(
        history_augmented_df["epoch"],
        history_augmented_df[
            validation_metric
        ],
        label="Validation"
    )

    axis.axvline(
        best_epoch,
        color="black",
        linestyle="--",
        linewidth=1,
        label=(
            f"Best epoch: {best_epoch}"
        )
    )

    axis.set_title(title)
    axis.set_xlabel("Epoch")
    axis.set_ylabel(y_label)
    axis.grid(alpha=0.2)
    axis.legend()

plt.show()

## 7. Validation-Based Threshold Selection

The best checkpoint is reloaded explicitly. The classification threshold is selected using only the validation partition by maximizing macro-F1, with balanced accuracy used as the tie-breaker.


In [ ]:
model_augmented = (
    tf.keras.models.load_model(
        CHECKPOINT_PATH
    )
)

assert (
    model_augmented.count_params()
    == 29121
)

print(
    "Best augmented checkpoint loaded."
)

print(
    "Checkpoint:",
    CHECKPOINT_PATH.resolve()
)

In [ ]:
validation_probabilities = (
    model_augmented
    .predict(
        validation_dataset,
        verbose=1
    )
    .reshape(-1)
)

assert (
    validation_probabilities.shape
    == y_validation.shape
)

assert np.isfinite(
    validation_probabilities
).all()

assert (
    validation_probabilities.min()
    >= 0.0
)

assert (
    validation_probabilities.max()
    <= 1.0
)

print(
    "Minimum validation probability:",
    float(
        validation_probabilities.min()
    )
)

print(
    "Maximum validation probability:",
    float(
        validation_probabilities.max()
    )
)

print(
    "Mean validation probability:",
    float(
        validation_probabilities.mean()
    )
)

print(
    "Predictions below 0.5:",
    int(
        np.sum(
            validation_probabilities
            < 0.5
        )
    )
)

print(
    "Predictions at or above 0.5:",
    int(
        np.sum(
            validation_probabilities
            >= 0.5
        )
    )
)

In [ ]:
threshold_records = []

for threshold in np.linspace(
    0.01,
    0.99,
    199
):
    validation_predictions = (
        validation_probabilities
        >= threshold
    ).astype(np.uint8)

    threshold_records.append({
        "threshold": float(
            threshold
        ),

        "accuracy": accuracy_score(
            y_validation,
            validation_predictions
        ),

        "balanced_accuracy":
            balanced_accuracy_score(
                y_validation,
                validation_predictions
            ),

        "macro_f1": f1_score(
            y_validation,
            validation_predictions,
            average="macro",
            zero_division=0
        ),

        "bird_recall": np.mean(
            validation_predictions[
                y_validation == 0
            ] == 0
        ),

        "drone_recall": np.mean(
            validation_predictions[
                y_validation == 1
            ] == 1
        )
    })

threshold_search_df = pd.DataFrame(
    threshold_records
)

best_threshold_row = (
    threshold_search_df
    .sort_values(
        [
            "macro_f1",
            "balanced_accuracy"
        ],
        ascending=False
    )
    .iloc[0]
)

optimal_threshold = float(
    best_threshold_row["threshold"]
)

print(
    "Selected validation threshold:",
    optimal_threshold
)

display(
    best_threshold_row
    .to_frame()
    .T
    .round(4)
)

In [ ]:
threshold_comparison_records = []

for threshold_name, threshold in [
    (
        "default_0.5",
        0.5
    ),
    (
        "validation_optimised",
        optimal_threshold
    )
]:
    validation_predictions = (
        validation_probabilities
        >= threshold
    ).astype(np.uint8)

    threshold_comparison_records.append({
        "threshold_name":
            threshold_name,

        "threshold":
            threshold,

        "accuracy":
            accuracy_score(
                y_validation,
                validation_predictions
            ),

        "balanced_accuracy":
            balanced_accuracy_score(
                y_validation,
                validation_predictions
            ),

        "macro_f1":
            f1_score(
                y_validation,
                validation_predictions,
                average="macro",
                zero_division=0
            ),

        "bird_recall":
            np.mean(
                validation_predictions[
                    y_validation == 0
                ] == 0
            ),

        "drone_recall":
            np.mean(
                validation_predictions[
                    y_validation == 1
                ] == 1
            )
    })

threshold_comparison_df = pd.DataFrame(
    threshold_comparison_records
)

display(
    threshold_comparison_df.round(4)
)

In [ ]:
LOCKED_THRESHOLD = float(
    optimal_threshold
)

assert np.isclose(
    LOCKED_THRESHOLD,
    0.6781818181818181
)

threshold_search_df.to_csv(
    OUTPUT_DIR
    / "validation_threshold_search.csv",
    index=False
)

threshold_comparison_df.to_csv(
    OUTPUT_DIR
    / "validation_threshold_comparison.csv",
    index=False
)

print(
    "Locked validation threshold:",
    LOCKED_THRESHOLD
)

print(
    "Validation-threshold results saved."
)

## 8. Held-Out Test Evaluation

The validation-optimized threshold is locked before the official test set is evaluated. The test partition is not used for training, checkpoint selection, or threshold optimization.


In [ ]:
test_probabilities = (
    model_augmented
    .predict(
        test_dataset,
        verbose=1
    )
    .reshape(-1)
)

assert (
    test_probabilities.shape
    == y_test.shape
)

assert np.isfinite(
    test_probabilities
).all()

assert test_probabilities.min() >= 0.0
assert test_probabilities.max() <= 1.0

test_predictions = (
    test_probabilities
    >= LOCKED_THRESHOLD
).astype(np.uint8)

print(
    "Minimum test probability:",
    float(
        test_probabilities.min()
    )
)

print(
    "Maximum test probability:",
    float(
        test_probabilities.max()
    )
)

print(
    "Mean test probability:",
    float(
        test_probabilities.mean()
    )
)

print(
    "Predicted birds:",
    int(
        np.sum(
            test_predictions == 0
        )
    )
)

print(
    "Predicted drones:",
    int(
        np.sum(
            test_predictions == 1
        )
    )
)

In [ ]:
test_report = classification_report(
    y_test,
    test_predictions,
    target_names=[
        "bird",
        "drone"
    ],
    output_dict=True,
    zero_division=0
)

test_metrics_df = pd.DataFrame([{
    "experiment":
        EXPERIMENT_NAME,

    "seed":
        RANDOM_SEED,

    "real_subset":
        REAL_SUBSET,

    "real_training_samples":
        len(X_real),

    "synthetic_training_samples":
        len(X_synthetic),

    "total_training_samples":
        len(X_train_augmented),

    "real_samples_per_class":
        int(
            np.sum(y_real == 0)
        ),

    "synthetic_samples_per_class":
        int(
            np.sum(y_synthetic == 0)
        ),

    "synthetic_to_real_ratio":
        SYNTHETIC_RATIO,

    "best_epoch":
        best_epoch,

    "best_validation_loss":
        best_validation_loss,

    "threshold":
        LOCKED_THRESHOLD,

    "accuracy":
        accuracy_score(
            y_test,
            test_predictions
        ),

    "balanced_accuracy":
        balanced_accuracy_score(
            y_test,
            test_predictions
        ),

    "macro_f1":
        f1_score(
            y_test,
            test_predictions,
            average="macro",
            zero_division=0
        ),

    "bird_precision":
        test_report[
            "bird"
        ]["precision"],

    "bird_recall":
        test_report[
            "bird"
        ]["recall"],

    "bird_f1":
        test_report[
            "bird"
        ]["f1-score"],

    "drone_precision":
        test_report[
            "drone"
        ]["precision"],

    "drone_recall":
        test_report[
            "drone"
        ]["recall"],

    "drone_f1":
        test_report[
            "drone"
        ]["f1-score"],

    "roc_auc":
        roc_auc_score(
            y_test,
            test_probabilities
        )
}])

display(
    test_metrics_df.round(4)
)

print(
    classification_report(
        y_test,
        test_predictions,
        target_names=[
            "bird",
            "drone"
        ],
        digits=4,
        zero_division=0
    )
)

In [ ]:
test_confusion_matrix = (
    confusion_matrix(
        y_test,
        test_predictions
    )
)

confusion_matrix_df = pd.DataFrame(
    test_confusion_matrix,
    index=[
        "True bird",
        "True drone"
    ],
    columns=[
        "Predicted bird",
        "Predicted drone"
    ]
)

display(
    confusion_matrix_df
)

plt.figure(figsize=(6, 5))

sns.heatmap(
    test_confusion_matrix,
    annot=True,
    fmt="d",
    cmap="Blues",
    xticklabels=[
        "Bird",
        "Drone"
    ],
    yticklabels=[
        "Bird",
        "Drone"
    ]
)

plt.title(
    "Test Confusion Matrix — "
    "10% Real + Synthetic 1:1"
)

plt.xlabel("Predicted target")
plt.ylabel("True target")
plt.tight_layout()
plt.show()

In [ ]:
test_results_df = metadata_test.copy()

test_results_df[
    "true_binary_label"
] = y_test

test_results_df[
    "drone_probability"
] = test_probabilities

test_results_df[
    "predicted_binary_label"
] = test_predictions

test_results_df[
    "true_target_group"
] = np.where(
    y_test == 1,
    "drone",
    "bird"
)

test_results_df[
    "predicted_target_group"
] = np.where(
    test_predictions == 1,
    "drone",
    "bird"
)

test_results_df[
    "correct"
] = (
    y_test
    == test_predictions
)

test_metrics_path = (
    OUTPUT_DIR
    / "test_metrics.csv"
)

test_predictions_path = (
    OUTPUT_DIR
    / "test_predictions.csv"
)

if test_metrics_path.exists():
    existing_metrics = pd.read_csv(
        test_metrics_path
    )

    assert np.isclose(
        existing_metrics.loc[0, "threshold"],
        LOCKED_THRESHOLD
    )

    assert np.isclose(
        existing_metrics.loc[0, "macro_f1"],
        test_metrics_df.loc[0, "macro_f1"]
    )

    print(
        "Existing test metrics verified; "
        "the file was not overwritten."
    )
else:
    test_metrics_df.to_csv(
        test_metrics_path,
        index=False
    )

    print("Test metrics saved.")

if test_predictions_path.exists():
    existing_predictions = pd.read_csv(
        test_predictions_path
    )

    assert len(existing_predictions) == len(
        test_results_df
    )

    assert np.allclose(
        existing_predictions[
            "drone_probability"
        ].to_numpy(),
        test_results_df[
            "drone_probability"
        ].to_numpy(),
        rtol=1e-5,
        atol=1e-6
    )

    print(
        "Existing test predictions verified; "
        "the file was not overwritten."
    )
else:
    test_results_df.to_csv(
        test_predictions_path,
        index=False
    )

    print("Test predictions saved.")


In [ ]:
experiment_configuration = {
    "experiment_name":
        EXPERIMENT_NAME,

    "random_seed":
        RANDOM_SEED,

    "real_subset":
        REAL_SUBSET,

    "generation_method": (
        synthetic_manifest[
            "generation_method"
        ]
    ),

    "real_training_samples":
        int(len(X_real)),

    "synthetic_training_samples":
        int(len(X_synthetic)),

    "total_training_samples":
        int(len(X_train_augmented)),

    "real_samples_per_class":
        int(np.sum(y_real == 0)),

    "synthetic_samples_per_class":
        int(np.sum(y_synthetic == 0)),

    "synthetic_to_real_ratio":
        SYNTHETIC_RATIO,

    "architecture_parameters":
        29121,

    "batch_size":
        BATCH_SIZE,

    "maximum_epochs":
        MAX_EPOCHS,

    "completed_epochs":
        int(len(history_augmented_df)),

    "best_epoch":
        int(best_epoch),

    "best_validation_loss":
        float(best_validation_loss),

    "decision_threshold":
        float(LOCKED_THRESHOLD),

    "threshold_selection": (
        "Maximum validation macro-F1 "
        "with balanced-accuracy tie-break"
    ),

    "validation_data":
        "Official real validation split",

    "test_data":
        "Official real test split"
}

with open(
    OUTPUT_DIR
    / "experiment_config.json",
    "w",
    encoding="utf-8"
) as file:
    json.dump(
        experiment_configuration,
        file,
        indent=4
    )

print(
    "Experiment configuration saved."
)

## 9. Aggregate Comparison with Real-Only Baselines

The augmented experiment is compared with the 10% and 25% real-only baselines under the same architecture, official partitions, and evaluation protocol.


In [ ]:
baseline_10_df = pd.read_csv(
    BASELINE_OUTPUT_DIR
    / "10_percent_seed_42"
    / "test_metrics.csv"
)

baseline_25_df = pd.read_csv(
    BASELINE_OUTPUT_DIR
    / "25_percent_seed_42"
    / "test_metrics.csv"
)

comparison_metrics = [
    "accuracy",
    "balanced_accuracy",
    "macro_f1",
    "bird_precision",
    "bird_recall",
    "bird_f1",
    "drone_recall",
    "roc_auc"
]

comparison_records = []

for model_name, metrics_df in [
    (
        "10% real-only",
        baseline_10_df
    ),
    (
        "10% real + synthetic 1:1",
        test_metrics_df
    ),
    (
        "25% real-only",
        baseline_25_df
    )
]:
    record = {
        "model": model_name
    }

    for metric in comparison_metrics:
        record[metric] = float(
            metrics_df.loc[
                0,
                metric
            ]
        )

    comparison_records.append(
        record
    )

comparison_df = pd.DataFrame(
    comparison_records
)

display(
    comparison_df.round(4)
)

In [ ]:
baseline_10_metrics = (
    comparison_df.iloc[0]
)

augmented_metrics = (
    comparison_df.iloc[1]
)

baseline_25_metrics = (
    comparison_df.iloc[2]
)

improvement_records = []

for metric in comparison_metrics:
    improvement_from_10 = (
        augmented_metrics[metric]
        - baseline_10_metrics[metric]
    )

    remaining_gap_to_25 = (
        augmented_metrics[metric]
        - baseline_25_metrics[metric]
    )

    full_10_to_25_gain = (
        baseline_25_metrics[metric]
        - baseline_10_metrics[metric]
    )

    if full_10_to_25_gain != 0:
        recovered_gap_percent = (
            improvement_from_10
            / full_10_to_25_gain
            * 100
        )
    else:
        recovered_gap_percent = np.nan

    improvement_records.append({
        "metric": metric,
        "improvement_over_10_percent":
            improvement_from_10,
        "difference_from_25_percent":
            remaining_gap_to_25,
        "recovered_10_to_25_gap_percent":
            recovered_gap_percent
    })

augmentation_improvement_df = (
    pd.DataFrame(
        improvement_records
    )
)

display(
    augmentation_improvement_df.round(4)
)

comparison_df.to_csv(
    OUTPUT_DIR
    / "baseline_comparison.csv",
    index=False
)

augmentation_improvement_df.to_csv(
    OUTPUT_DIR
    / "augmentation_improvement.csv",
    index=False
)

print(
    "Baseline comparison results saved."
)

### Aggregate interpretation

Adding one transformation-based synthetic child per real training sample substantially improved classification performance compared with the 10% real-only baseline. The augmented model increased test macro-F1 from 0.8082 to 0.9378 and balanced accuracy from 0.8497 to 0.9368, recovering 86.6% and 84.9%, respectively, of the performance gap between the 10% and 25% real-data baselines. The largest improvement occurred for the minority bird class, whose F1-score increased from 0.6797 to 0.8934. However, the augmented model remained below the 25% real-only baseline, indicating that transformation-based augmentation provides substantial but not complete replacement for additional independent real observations.

## 10. Performance by Original Target Subtype

This section evaluates whether the augmented classifier performs consistently across the original bird species and drone models. Results for groups with very few test samples must be interpreted cautiously.

In [ ]:
required_subtype_columns = [
    "original_label",
    "true_target_group",
    "correct",
    "drone_probability"
]

missing_subtype_columns = [
    column
    for column in required_subtype_columns
    if column not in test_results_df.columns
]

if missing_subtype_columns:
    raise KeyError(
        "Missing columns required for subtype analysis: "
        + ", ".join(missing_subtype_columns)
    )

subtype_metrics_augmented = (
    test_results_df
    .groupby(
        [
            "true_target_group",
            "original_label"
        ],
        observed=True
    )
    .agg(
        samples=("correct", "size"),
        correct_predictions=("correct", "sum"),
        recall=("correct", "mean"),
        mean_drone_probability=(
            "drone_probability",
            "mean"
        ),
        median_drone_probability=(
            "drone_probability",
            "median"
        )
    )
    .reset_index()
)

subtype_metrics_augmented[
    "correct_predictions"
] = subtype_metrics_augmented[
    "correct_predictions"
].astype(int)

subtype_metrics_augmented = (
    subtype_metrics_augmented
    .sort_values(
        [
            "true_target_group",
            "recall",
            "samples"
        ],
        ascending=[
            True,
            True,
            False
        ]
    )
    .reset_index(drop=True)
)

subtype_metrics_augmented.to_csv(
    OUTPUT_DIR / "subtype_metrics.csv",
    index=False
)

subtype_display = (
    subtype_metrics_augmented.copy()
)

subtype_display["recall"] = (
    subtype_display["recall"]
    .map(lambda value: f"{value:.2%}")
)

display(
    subtype_display.style.format({
        "mean_drone_probability": "{:.4f}",
        "median_drone_probability": "{:.4f}"
    })
)

print(
    "Subtype metrics saved:",
    OUTPUT_DIR / "subtype_metrics.csv"
)

In [ ]:
subtype_plot_df = (
    subtype_metrics_augmented.copy()
)

subtype_plot_df["display_label"] = (
    subtype_plot_df["original_label"]
    + " (n="
    + subtype_plot_df["samples"].astype(str)
    + ")"
)

plt.figure(figsize=(11, 6))

axis = sns.barplot(
    data=subtype_plot_df,
    x="recall",
    y="display_label",
    hue="true_target_group",
    palette={
        "bird": "tab:blue",
        "drone": "tab:orange"
    },
    dodge=False
)

plt.axvline(
    0.90,
    color="black",
    linestyle="--",
    linewidth=1,
    label="90% recall reference"
)

plt.xlim(0.0, 1.02)
plt.xlabel("Subtype recall")
plt.ylabel("Original target subtype")
plt.title(
    "Test Recall by Original Target Subtype\n"
    "10% Real + Transformation-Based Synthetic Data (1:1)"
)

handles, labels = axis.get_legend_handles_labels()

unique_legend = dict(zip(labels, handles))

axis.legend(
    unique_legend.values(),
    unique_legend.keys(),
    title="Target group",
    loc="lower right"
)

plt.tight_layout()
plt.show()

## 11. Performance by Target Range

The test set is divided into four target-range intervals to examine whether classification performance changes with distance. Because the class distribution varies across intervals, balanced accuracy, macro-F1, and class-specific recall are more informative than accuracy alone.

In [ ]:
if "range_m" not in test_results_df.columns:
    raise KeyError(
        "The metadata does not contain the required "
        "'range_m' column."
    )

RANGE_BINS = [
    0.0,
    30.0,
    50.0,
    75.0,
    np.inf
]

RANGE_LABELS = [
    "<30 m",
    "30–50 m",
    "50–75 m",
    ">75 m"
]

test_results_df["range_bin"] = pd.cut(
    test_results_df["range_m"],
    bins=RANGE_BINS,
    labels=RANGE_LABELS,
    right=False,
    include_lowest=True
)

assert test_results_df["range_bin"].notna().all()

range_metric_records = []

for range_label in RANGE_LABELS:
    range_subset = test_results_df[
        test_results_df["range_bin"]
        == range_label
    ]

    y_range_true = range_subset[
        "true_binary_label"
    ].to_numpy()

    y_range_predicted = range_subset[
        "predicted_binary_label"
    ].to_numpy()

    range_probabilities = range_subset[
        "drone_probability"
    ].to_numpy()

    class_counts = np.bincount(
        y_range_true,
        minlength=2
    )

    if len(np.unique(y_range_true)) == 2:
        range_roc_auc = roc_auc_score(
            y_range_true,
            range_probabilities
        )
    else:
        range_roc_auc = np.nan

    range_metric_records.append({
        "range_bin":
            range_label,

        "samples":
            len(range_subset),

        "bird_samples":
            int(class_counts[0]),

        "drone_samples":
            int(class_counts[1]),

        "accuracy":
            accuracy_score(
                y_range_true,
                y_range_predicted
            ),

        "balanced_accuracy":
            balanced_accuracy_score(
                y_range_true,
                y_range_predicted
            ),

        "macro_f1":
            f1_score(
                y_range_true,
                y_range_predicted,
                average="macro",
                zero_division=0
            ),

        "bird_recall":
            recall_score(
                y_range_true,
                y_range_predicted,
                pos_label=0,
                zero_division=0
            ),

        "drone_recall":
            recall_score(
                y_range_true,
                y_range_predicted,
                pos_label=1,
                zero_division=0
            ),

        "roc_auc":
            range_roc_auc,

        "mean_drone_probability":
            float(
                range_probabilities.mean()
            )
    })

range_metrics_augmented = pd.DataFrame(
    range_metric_records
)

range_metrics_augmented.to_csv(
    OUTPUT_DIR / "range_metrics.csv",
    index=False
)

display(
    range_metrics_augmented.style.format({
        "accuracy": "{:.4f}",
        "balanced_accuracy": "{:.4f}",
        "macro_f1": "{:.4f}",
        "bird_recall": "{:.4f}",
        "drone_recall": "{:.4f}",
        "roc_auc": "{:.4f}",
        "mean_drone_probability": "{:.4f}"
    })
)

print(
    "Range metrics saved:",
    OUTPUT_DIR / "range_metrics.csv"
)

In [ ]:
range_plot_df = (
    range_metrics_augmented[
        [
            "range_bin",
            "balanced_accuracy",
            "macro_f1",
            "bird_recall",
            "drone_recall"
        ]
    ]
    .melt(
        id_vars="range_bin",
        var_name="metric",
        value_name="score"
    )
)

metric_names = {
    "balanced_accuracy":
        "Balanced accuracy",
    "macro_f1":
        "Macro-F1",
    "bird_recall":
        "Bird recall",
    "drone_recall":
        "Drone recall"
}

range_plot_df["metric"] = (
    range_plot_df["metric"]
    .map(metric_names)
)

plt.figure(figsize=(11, 6))

sns.lineplot(
    data=range_plot_df,
    x="range_bin",
    y="score",
    hue="metric",
    marker="o",
    linewidth=2
)

plt.ylim(0.0, 1.02)
plt.xlabel("Target-range interval")
plt.ylabel("Test score")
plt.title(
    "Test Performance by Target Range\n"
    "10% Real + Transformation-Based Synthetic Data (1:1)"
)
plt.legend(
    title="Metric",
    loc="lower left"
)
plt.grid(alpha=0.25)
plt.tight_layout()
plt.show()

### Augmented-model observations

- Heron is the weakest bird subtype, with 73.33% recall; seagull recall is 84.82%.
- D1 and D3 are the most difficult drone models, although both remain above 94% recall.
- Pigeon and raven results are descriptive only because their test supports are four and one samples, respectively.
- Performance decreases with target range. Beyond 75 m, balanced accuracy is 0.8677, macro-F1 is 0.7757, bird recall is 0.8039, and drone recall is 0.9315.
- The low long-range macro-F1 reflects the imbalanced range subset and the remaining difficulty of minority-class bird recognition.


## 12. Subtype and Range Comparison with Real-Only Baselines

The augmented model is compared with the 10% and 25% real-only baselines using the same official test set. This comparison determines whether synthetic augmentation improves robustness uniformly or only for particular target subtypes and range intervals.

In [ ]:
baseline_analysis_paths = {
    "10% real-only": (
        BASELINE_OUTPUT_DIR
        / "10_percent_seed_42"
    ),
    "25% real-only": (
        BASELINE_OUTPUT_DIR
        / "25_percent_seed_42"
    )
}

required_analysis_files = []

for model_name, model_directory in (
    baseline_analysis_paths.items()
):
    required_analysis_files.extend([
        model_directory
        / "subtype_metrics.csv",

        model_directory
        / "range_metrics.csv"
    ])

missing_analysis_files = [
    path
    for path in required_analysis_files
    if not path.exists()
]

if missing_analysis_files:
    raise FileNotFoundError(
        "Missing baseline analysis files:\n"
        + "\n".join(
            str(path.resolve())
            for path in missing_analysis_files
        )
    )

print(
    "All baseline subtype and range "
    "analysis files were found."
)

In [ ]:
def standardize_subtype_columns(
    dataframe,
    model_name
):
    standardized_df = dataframe.copy()

    if "recall" not in standardized_df.columns:
        if (
            "classification_recall"
            in standardized_df.columns
        ):
            standardized_df = (
                standardized_df.rename(
                    columns={
                        "classification_recall":
                            "recall"
                    }
                )
            )
        else:
            raise KeyError(
                f"No recall column found for "
                f"{model_name}. Available columns: "
                f"{standardized_df.columns.tolist()}"
            )

    standardized_df["recall"] = (
        pd.to_numeric(
            standardized_df["recall"],
            errors="raise"
        )
        .astype(float)
    )

    standardized_df["original_label"] = (
        standardized_df["original_label"]
        .astype(str)
        .str.strip()
    )

    standardized_df[
        "true_target_group"
    ] = (
        standardized_df[
            "true_target_group"
        ]
        .astype(str)
        .str.strip()
        .str.lower()
    )

    return standardized_df


subtype_comparison_frames = []

for model_name, model_directory in (
    baseline_analysis_paths.items()
):
    model_subtype_df = pd.read_csv(
        model_directory
        / "subtype_metrics.csv"
    )

    model_subtype_df = (
        standardize_subtype_columns(
            model_subtype_df,
            model_name
        )
    )

    model_subtype_df.insert(
        0,
        "model",
        model_name
    )

    subtype_comparison_frames.append(
        model_subtype_df
    )

augmented_subtype_comparison = (
    standardize_subtype_columns(
        subtype_metrics_augmented,
        "10% real + synthetic 1:1"
    )
)

augmented_subtype_comparison.insert(
    0,
    "model",
    "10% real + synthetic 1:1"
)

subtype_comparison_frames.append(
    augmented_subtype_comparison
)

subtype_comparison_df = pd.concat(
    subtype_comparison_frames,
    ignore_index=True
)

expected_models = [
    "10% real-only",
    "10% real + synthetic 1:1",
    "25% real-only"
]

print(
    "Models available:",
    subtype_comparison_df[
        "model"
    ].unique().tolist()
)

subtype_recall_comparison = (
    subtype_comparison_df
    .pivot_table(
        index=[
            "true_target_group",
            "original_label"
        ],
        columns="model",
        values="recall",
        aggfunc="first",
        observed=True
    )
    .reset_index()
)

subtype_recall_comparison.columns.name = None

missing_model_columns = [
    model_name
    for model_name in expected_models
    if model_name
    not in subtype_recall_comparison.columns
]

if missing_model_columns:
    raise KeyError(
        "Missing model columns after pivot: "
        + ", ".join(missing_model_columns)
    )

sample_counts = (
    subtype_comparison_df
    .groupby(
        [
            "true_target_group",
            "original_label"
        ],
        observed=True
    )["samples"]
    .first()
    .reset_index()
)

subtype_recall_comparison = (
    sample_counts.merge(
        subtype_recall_comparison,
        on=[
            "true_target_group",
            "original_label"
        ],
        how="left",
        validate="one_to_one"
    )
)

subtype_recall_comparison[
    "augmentation_change_vs_10_percent"
] = (
    subtype_recall_comparison[
        "10% real + synthetic 1:1"
    ]
    - subtype_recall_comparison[
        "10% real-only"
    ]
)

subtype_recall_comparison[
    "augmentation_difference_from_25_percent"
] = (
    subtype_recall_comparison[
        "10% real + synthetic 1:1"
    ]
    - subtype_recall_comparison[
        "25% real-only"
    ]
)

subtype_recall_comparison = (
    subtype_recall_comparison
    .sort_values(
        [
            "true_target_group",
            "augmentation_change_vs_10_percent"
        ],
        ascending=[
            True,
            False
        ]
    )
    .reset_index(drop=True)
)

subtype_comparison_df.to_csv(
    OUTPUT_DIR
    / "subtype_baseline_comparison_long.csv",
    index=False
)

subtype_recall_comparison.to_csv(
    OUTPUT_DIR
    / "subtype_recall_comparison.csv",
    index=False
)

display(
    subtype_recall_comparison.style.format({
        "10% real-only": "{:.4f}",
        "10% real + synthetic 1:1": "{:.4f}",
        "25% real-only": "{:.4f}",
        "augmentation_change_vs_10_percent":
            "{:+.4f}",
        "augmentation_difference_from_25_percent":
            "{:+.4f}"
    })
)

In [ ]:
model_order = [
    "10% real-only",
    "10% real + synthetic 1:1",
    "25% real-only"
]

subtype_comparison_df["model"] = pd.Categorical(
    subtype_comparison_df["model"],
    categories=model_order,
    ordered=True
)

subtype_comparison_df["display_label"] = (
    subtype_comparison_df["original_label"]
    + " (n="
    + subtype_comparison_df["samples"].astype(str)
    + ")"
)

subtype_comparison_df = (
    subtype_comparison_df
    .sort_values([
        "true_target_group",
        "original_label",
        "model"
    ])
)

plt.figure(figsize=(13, 8))

sns.barplot(
    data=subtype_comparison_df,
    x="recall",
    y="display_label",
    hue="model",
    hue_order=model_order
)

plt.axvline(
    0.90,
    color="black",
    linestyle="--",
    linewidth=1,
    label="90% recall reference"
)

plt.xlim(0.0, 1.02)
plt.xlabel("Test recall")
plt.ylabel("Original target subtype")
plt.title(
    "Subtype Recall: Real-Only versus "
    "Synthetic-Augmented Training"
)
plt.legend(
    title="Training configuration",
    loc="lower right"
)
plt.tight_layout()
plt.show()

In [ ]:
range_comparison_frames = []

for model_name, model_directory in (
    baseline_analysis_paths.items()
):
    model_range_df = pd.read_csv(
        model_directory
        / "range_metrics.csv"
    )

    model_range_df.insert(
        0,
        "model",
        model_name
    )

    range_comparison_frames.append(
        model_range_df
    )

augmented_range_comparison = (
    range_metrics_augmented.copy()
)

augmented_range_comparison.insert(
    0,
    "model",
    "10% real + synthetic 1:1"
)

range_comparison_frames.append(
    augmented_range_comparison
)

range_comparison_df = pd.concat(
    range_comparison_frames,
    ignore_index=True
)

range_comparison_df["range_bin"] = (
    pd.Categorical(
        range_comparison_df["range_bin"],
        categories=RANGE_LABELS,
        ordered=True
    )
)

range_comparison_df["model"] = pd.Categorical(
    range_comparison_df["model"],
    categories=model_order,
    ordered=True
)

range_comparison_df = (
    range_comparison_df
    .sort_values([
        "range_bin",
        "model"
    ])
    .reset_index(drop=True)
)

range_comparison_df.to_csv(
    OUTPUT_DIR
    / "range_baseline_comparison.csv",
    index=False
)

display(
    range_comparison_df[
        [
            "model",
            "range_bin",
            "samples",
            "bird_samples",
            "drone_samples",
            "balanced_accuracy",
            "macro_f1",
            "bird_recall",
            "drone_recall",
            "roc_auc"
        ]
    ].style.format({
        "balanced_accuracy": "{:.4f}",
        "macro_f1": "{:.4f}",
        "bird_recall": "{:.4f}",
        "drone_recall": "{:.4f}",
        "roc_auc": "{:.4f}"
    })
)

In [ ]:
range_metrics_to_plot = {
    "balanced_accuracy":
        "Balanced Accuracy",
    "macro_f1":
        "Macro-F1",
    "bird_recall":
        "Bird Recall"
}

fig, axes = plt.subplots(
    1,
    3,
    figsize=(17, 5),
    sharey=True,
    constrained_layout=True
)

for axis, (
    metric_column,
    metric_title
) in zip(
    axes,
    range_metrics_to_plot.items()
):
    sns.lineplot(
        data=range_comparison_df,
        x="range_bin",
        y=metric_column,
        hue="model",
        hue_order=model_order,
        marker="o",
        linewidth=2,
        ax=axis
    )

    axis.set_title(metric_title)
    axis.set_xlabel("Target-range interval")
    axis.set_ylabel("Test score")
    axis.set_ylim(0.0, 1.02)
    axis.grid(alpha=0.25)

    if axis is not axes[0]:
        axis.get_legend().remove()

axes[0].legend(
    title="Training configuration",
    loc="lower left"
)

fig.suptitle(
    "Performance by Target Range: "
    "Real-Only versus Synthetic-Augmented Training",
    fontsize=14
)

plt.show()

In [ ]:
range_metric_columns = [
    "balanced_accuracy",
    "macro_f1",
    "bird_recall",
    "drone_recall",
    "roc_auc"
]

range_change_records = []

for range_label in RANGE_LABELS:
    range_rows = (
        range_comparison_df[
            range_comparison_df["range_bin"]
            == range_label
        ]
        .set_index("model")
    )

    for metric_name in range_metric_columns:
        augmented_value = float(
            range_rows.loc[
                "10% real + synthetic 1:1",
                metric_name
            ]
        )

        baseline_10_value = float(
            range_rows.loc[
                "10% real-only",
                metric_name
            ]
        )

        baseline_25_value = float(
            range_rows.loc[
                "25% real-only",
                metric_name
            ]
        )

        range_change_records.append({
            "range_bin":
                range_label,

            "metric":
                metric_name,

            "improvement_over_10_percent":
                augmented_value
                - baseline_10_value,

            "difference_from_25_percent":
                augmented_value
                - baseline_25_value
        })

range_improvement_df = pd.DataFrame(
    range_change_records
)

range_improvement_df.to_csv(
    OUTPUT_DIR
    / "range_improvement_summary.csv",
    index=False
)

range_improvement_pivot = (
    range_improvement_df
    .pivot(
        index="range_bin",
        columns="metric",
        values="improvement_over_10_percent"
    )
    .reindex(RANGE_LABELS)
    .reset_index()
)

range_improvement_pivot.columns.name = None

display(
    range_improvement_pivot.style.format({
        "balanced_accuracy": "{:+.4f}",
        "macro_f1": "{:+.4f}",
        "bird_recall": "{:+.4f}",
        "drone_recall": "{:+.4f}",
        "roc_auc": "{:+.4f}"
    })
)

print(
    "Positive values indicate improvement "
    "over the 10% real-only baseline."
)

## 13. Discussion, Limitations, and Conclusion

### Overall effect of synthetic augmentation

Adding one transformation-based synthetic child for each sample in the 10% real-data subset substantially improved test performance. The augmented model achieved an accuracy of **0.9697**, balanced accuracy of **0.9368**, macro-F1 of **0.9378**, and ROC-AUC of **0.9926**. In comparison, the 10% real-only baseline achieved an accuracy of 0.8942, balanced accuracy of 0.8497, macro-F1 of 0.8082, and ROC-AUC of 0.9353.

Therefore, augmentation increased macro-F1 by **0.1297** and balanced accuracy by **0.0871**. It recovered approximately **86.6% of the macro-F1 gap** and **84.9% of the balanced-accuracy gap** between the 10% and 25% real-only baselines.

### Minority-class performance

The principal benefit appeared in bird classification. Bird precision increased from 0.5979 to 0.8961, bird recall increased from 0.7874 to 0.8907, and bird F1 increased from 0.6797 to 0.8934. Drone recall also improved from 0.9120 to 0.9828. These results indicate that augmentation reduced the strong class-dependent errors observed in the 10% real-only model.

### Subtype-dependent effects

The benefits were not uniform across all target subtypes. Recall improved substantially for black-headed gulls, seagulls, D1, and D3. D1 showed the largest reliable subtype improvement, increasing from 0.7020 to 0.9440 recall.

In contrast, heron recall decreased from 0.9333 to 0.7333, and the mixed seagull and black-headed-gull group decreased slightly from 0.9333 to 0.9111. Results for pigeons and ravens should not be interpreted as evidence of generalization because these groups contain only four and one test samples, respectively.

### Performance across target range

Synthetic augmentation improved balanced accuracy, macro-F1, and ROC-AUC across all range intervals. The largest practical correction occurred beyond 75 m, where macro-F1 increased from 0.4585 to 0.7757 and drone recall increased from 0.5179 to 0.9315.

At this range, bird recall decreased from 0.9804 to 0.8039. The high bird recall of the 10% baseline was accompanied by very poor drone recall, demonstrating a strong prediction bias rather than balanced long-range performance. The augmented model produced a substantially more balanced result, although long-range classification remained the most difficult operating condition.

### Comparison with additional real data

The augmented model approached but did not surpass the 25% real-only baseline. Its macro-F1 was lower by 0.0201, balanced accuracy by approximately 0.0155, and bird recall by approximately 0.0231. This shows that the transformation-based synthetic data replaced a substantial portion, but not all, of the benefit obtained from additional independent real observations.

### Limitations

This experiment has several limitations:

1. The results are based on a single random seed.
2. The official split is segment-based and may not measure session-independent generalization.
3. Synthetic children remain correlated with their real parent samples.
4. The 2,300 training observations originate from only 1,150 independent real parent samples.
5. Some bird subtypes have very small test support.
6. The validation-optimized threshold is specific to this trained model.
7. The augmentation parameters and the 1:1 synthetic-to-real ratio were not compared with alternative configurations.

### Conclusion

Transformation-based synthetic augmentation is effective in the low-data setting. It substantially improves overall discrimination, minority-class performance, and robustness across target ranges compared with the 10% real-only baseline. However, it does not completely reproduce the benefit of collecting additional independent real data, and subtype-specific degradation—particularly for herons—shows that augmentation quality must be evaluated at a more detailed level than aggregate accuracy alone.

The results support using this augmentation strategy as a data-efficiency technique, while motivating further experiments with multiple seeds, alternative augmentation ratios, and session-independent evaluation.